# Anomaly Detection Model Training
This notebook trains anomaly detection models using the combined IDS2017 dataset.
It uses the pre-trained scaler.pkl and pca.pkl for preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Combine Datasets

In [ ]:
files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

def dataset_file_to_df(index: int, dataset_files: list):
    read_file = f"datasets/CSVs/{dataset_files[index]}"
    df_read = pd.read_csv(read_file)
    df_read.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
    return df_read

dfs = [dataset_file_to_df(i, files) for i in range(len(files))]
df = pd.concat(dfs, axis=0, ignore_index=True)

print(f"Combined dataset shape: {df.shape}")
print(f"\nLabel distribution:")
print(df['Label'].value_counts())

## 2. Preprocessing

In [ ]:
def normalize(df):
    """Normalize numeric columns based on skewness."""
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    df_norm = df.copy()
    for col in numeric_columns:
        if df[col].skew() > 0 or col == "Source Port":
            df_norm[col] = np.log1p(df[col].clip(lower=-0.99))
        elif df[col].skew() < 0:
            df_norm[col] = df[col] ** 2
    return df_norm

df_norm = normalize(df)
print("Normalization complete.")

In [ ]:
# Separate benign (normal) and malicious (anomaly) data
df_benign = df_norm[df_norm['Label'] == 'BENIGN']
df_malicious = df_norm[df_norm['Label'] != 'BENIGN']

print(f"Benign samples: {len(df_benign)}")
print(f"Malicious samples: {len(df_malicious)}")

# Columns to drop
drop_cols = ["Label", "Flow ID", "Src IP", "Timestamp", "Dst IP"]

X_benign = df_benign.drop(drop_cols, axis=1)
X_malicious = df_malicious.drop(drop_cols, axis=1)

print(f"\nFeatures: {X_benign.shape[1]}")

In [ ]:
# Split benign data into train and test
X_train_benign, X_test_benign = train_test_split(X_benign, test_size=0.3, random_state=42)

# Sample malicious data for testing (to balance evaluation)
X_test_malicious = X_malicious.sample(n=min(len(X_test_benign), len(X_malicious)), random_state=42)

print(f"Train (benign only): {len(X_train_benign)}")
print(f"Test benign: {len(X_test_benign)}")
print(f"Test malicious: {len(X_test_malicious)}")

## 3. Load Pre-trained Scaler and PCA

In [ ]:
# Load pre-trained scaler and PCA
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('pca.pkl', 'rb') as f:
    pca = pickle.load(f)

print(f"Scaler expects {scaler.n_features_in_} features")
print(f"PCA reduces to {pca.n_components_} components")

In [ ]:
# Apply scaler and PCA
X_train_scaled = scaler.transform(X_train_benign)
X_test_benign_scaled = scaler.transform(X_test_benign)
X_test_malicious_scaled = scaler.transform(X_test_malicious)

X_train_pca = pca.transform(X_train_scaled)
X_test_benign_pca = pca.transform(X_test_benign_scaled)
X_test_malicious_pca = pca.transform(X_test_malicious_scaled)

print(f"Train PCA shape: {X_train_pca.shape}")
print(f"Test benign PCA shape: {X_test_benign_pca.shape}")
print(f"Test malicious PCA shape: {X_test_malicious_pca.shape}")

## 4. Train Anomaly Detection Models

In [ ]:
def evaluate_anomaly_model(model, X_normal, X_anomaly, model_name):
    """Evaluate anomaly detection model.
    Normal data should be predicted as 1 (inlier).
    Anomaly data should be predicted as -1 (outlier).
    """
    y_pred_normal = model.predict(X_normal)
    y_pred_anomaly = model.predict(X_anomaly)
    
    # Calculate metrics
    normal_accuracy = np.sum(y_pred_normal == 1) / len(y_pred_normal)
    anomaly_detection_rate = np.sum(y_pred_anomaly == -1) / len(y_pred_anomaly)
    
    print(f"\n=== {model_name} ===")
    print(f"Normal traffic correctly identified: {normal_accuracy:.4f} ({np.sum(y_pred_normal == 1)}/{len(y_pred_normal)})")
    print(f"Anomaly detection rate: {anomaly_detection_rate:.4f} ({np.sum(y_pred_anomaly == -1)}/{len(y_pred_anomaly)})")
    print(f"Combined score: {(normal_accuracy + anomaly_detection_rate) / 2:.4f}")
    
    return normal_accuracy, anomaly_detection_rate

### 4.1 One-Class SVM

In [ ]:
from itertools import product

class OneClassSVMGridSearch:
    def __init__(self, param_grid):
        self.param_grid = param_grid
        self.best_params_ = None
        self.best_score_ = -np.inf
        self.best_model_ = None

    def fit(self, X_train, X_normal, X_anomaly):
        param_names = list(self.param_grid.keys())
        param_values = list(self.param_grid.values())

        for param_comb in product(*param_values):
            params = dict(zip(param_names, param_comb))
            
            model = OneClassSVM(**params)
            model.fit(X_train)

            y_pred_normal = model.predict(X_normal)
            y_pred_anomaly = model.predict(X_anomaly)

            normal_acc = np.sum(y_pred_normal == 1) / len(y_pred_normal)
            anomaly_acc = np.sum(y_pred_anomaly == -1) / len(y_pred_anomaly)
            score = normal_acc + anomaly_acc

            if score > self.best_score_:
                self.best_score_ = score
                self.best_params_ = params
                self.best_model_ = model

        return self

# Grid search for OneClassSVM
param_grid_ocsvm = {
    "nu": [0.01, 0.05, 0.1],
    "kernel": ["rbf"],
    "gamma": ["scale", 0.01, 0.1, 1]
}

print("Running GridSearch for OneClassSVM...")
search_ocsvm = OneClassSVMGridSearch(param_grid_ocsvm)
search_ocsvm.fit(X_train_pca, X_test_benign_pca, X_test_malicious_pca)

print(f"\nBest parameters: {search_ocsvm.best_params_}")
print(f"Best score: {search_ocsvm.best_score_:.4f}")

In [ ]:
# Train final OneClassSVM with best params
ocsvm = search_ocsvm.best_model_
evaluate_anomaly_model(ocsvm, X_test_benign_pca, X_test_malicious_pca, "One-Class SVM")

### 4.2 Isolation Forest

In [ ]:
class IsolationForestGridSearch:
    def __init__(self, param_grid):
        self.param_grid = param_grid
        self.best_params_ = None
        self.best_score_ = -np.inf
        self.best_model_ = None

    def fit(self, X_train, X_normal, X_anomaly):
        param_names = list(self.param_grid.keys())
        param_values = list(self.param_grid.values())

        for param_comb in product(*param_values):
            params = dict(zip(param_names, param_comb))
            
            model = IsolationForest(**params, random_state=42)
            model.fit(X_train)

            y_pred_normal = model.predict(X_normal)
            y_pred_anomaly = model.predict(X_anomaly)

            normal_acc = np.sum(y_pred_normal == 1) / len(y_pred_normal)
            anomaly_acc = np.sum(y_pred_anomaly == -1) / len(y_pred_anomaly)
            score = normal_acc + anomaly_acc

            if score > self.best_score_:
                self.best_score_ = score
                self.best_params_ = params
                self.best_model_ = model

        return self

# Grid search for IsolationForest
param_grid_iforest = {
    "n_estimators": [100, 200],
    "max_samples": ["auto", 0.5],
    "contamination": [0.01, 0.05, 0.1]
}

print("Running GridSearch for Isolation Forest...")
search_iforest = IsolationForestGridSearch(param_grid_iforest)
search_iforest.fit(X_train_pca, X_test_benign_pca, X_test_malicious_pca)

print(f"\nBest parameters: {search_iforest.best_params_}")
print(f"Best score: {search_iforest.best_score_:.4f}")

In [ ]:
# Evaluate Isolation Forest
iforest = search_iforest.best_model_
evaluate_anomaly_model(iforest, X_test_benign_pca, X_test_malicious_pca, "Isolation Forest")

### 4.3 Local Outlier Factor (for comparison)

In [ ]:
# LOF with novelty detection mode
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.1, novelty=True)
lof.fit(X_train_pca)

evaluate_anomaly_model(lof, X_test_benign_pca, X_test_malicious_pca, "Local Outlier Factor")

## 5. Model Comparison

In [ ]:
# Compare all models
models = {
    'One-Class SVM': ocsvm,
    'Isolation Forest': iforest,
    'Local Outlier Factor': lof
}

results = []
for name, model in models.items():
    y_pred_normal = model.predict(X_test_benign_pca)
    y_pred_anomaly = model.predict(X_test_malicious_pca)
    
    normal_acc = np.sum(y_pred_normal == 1) / len(y_pred_normal)
    anomaly_rate = np.sum(y_pred_anomaly == -1) / len(y_pred_anomaly)
    
    results.append({
        'Model': name,
        'Normal Accuracy': normal_acc,
        'Anomaly Detection Rate': anomaly_rate,
        'Combined Score': (normal_acc + anomaly_rate) / 2
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results_df))
width = 0.35

bars1 = ax.bar(x - width/2, results_df['Normal Accuracy'], width, label='Normal Accuracy')
bars2 = ax.bar(x + width/2, results_df['Anomaly Detection Rate'], width, label='Anomaly Detection Rate')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Anomaly Detection Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'])
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

## 6. Save Best Models

In [ ]:
# Find best model
best_model_name = results_df.loc[results_df['Combined Score'].idxmax(), 'Model']
best_model = models[best_model_name]

print(f"Best model: {best_model_name}")

# Save models
with open('oneclass_svm.pkl', 'wb') as f:
    pickle.dump(ocsvm, f)
print("Saved: oneclass_svm.pkl")

with open('isolation_forest.pkl', 'wb') as f:
    pickle.dump(iforest, f)
print("Saved: isolation_forest.pkl")

with open('lof.pkl', 'wb') as f:
    pickle.dump(lof, f)
print("Saved: lof.pkl")

# Save the best model with a generic name
with open('anomaly_detector.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f"\nSaved best model as: anomaly_detector.pkl")

## 7. Test on New Data Example

In [ ]:
def predict_anomaly(sample, scaler, pca, model):
    """Predict if a sample is normal (1) or anomaly (-1)."""
    sample_scaled = scaler.transform(sample)
    sample_pca = pca.transform(sample_scaled)
    prediction = model.predict(sample_pca)
    return prediction

# Test on a few samples
test_normal = X_test_benign.iloc[:5]
test_anomaly = X_test_malicious.iloc[:5]

print("Testing normal samples:")
pred_normal = predict_anomaly(test_normal, scaler, pca, best_model)
print(f"Predictions: {pred_normal} (expected: [1, 1, 1, 1, 1])")

print("\nTesting anomaly samples:")
pred_anomaly = predict_anomaly(test_anomaly, scaler, pca, best_model)
print(f"Predictions: {pred_anomaly} (expected: [-1, -1, -1, -1, -1])")